# Challenge Day (4/21)

End-to-end workflow for the in-class identification challenge. The TA hands out 5 unknown feature vectors (a mix of 512-d ArcFace and 1000-d VGG19 pre-softmax logits). We take 1–3 photos per classmate, drop them into Drive as `{N}Gallery/` folders, rebuild the gallery, then run each unknown through the matcher that fits its dimensionality.

**Runtime:** set **Runtime → Change runtime type → GPU (T4)**.

All source patches from challenge day are already merged into `main` — numeric folder stems (`1Gallery` … `38Gallery`) parse automatically, the gallery walker picks up both `Gallery/` and `Gallery /`, and the loaders handle TA's pickled dict-wrapped `.npy` format.

## 1. Clone repo and install

In [ ]:
GITHUB_URL = 'https://github.com/ekw4tu/DS3001-Final.git'

!git clone $GITHUB_URL /content/DS3001-Final 2>/dev/null || (cd /content/DS3001-Final && git pull)
%cd /content/DS3001-Final
!pip install -q -r requirements.txt
!pip uninstall -y -q onnxruntime && pip install -q onnxruntime-gpu

## 2. Mount Drive

`FR_DATA_ROOT` points at the shared picture folder so the pipeline reads straight from Drive. Classmates added during class live as `{N}Gallery/` siblings of the original celebrity folders.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.environ['FR_DATA_ROOT'] = '/content/drive/MyDrive/DS_NN_Project_Pictures'
os.environ['FR_ARTIFACTS_DIR'] = '/content/drive/MyDrive/facial_recognition_artifacts'
!ls "$FR_DATA_ROOT"

## 3. Build the gallery (celebrities + classmates)

Picks up every `{N}Gallery/` classmate folder alongside the 10 original celebrities. Expect ~180 vectors across ~48 identities once the room is photographed.

In [ ]:
!python scripts/build_gallery.py --gpu

## 4. Receive the unknowns

Upload the `.npy` files the TA hands out to `/content/` (Colab file panel → Upload). Then sniff each one's dimensionality so we know which matcher to use:

- **512-d** → ArcFace → `identify_unknowns.py`
- **1000-d** → VGG19 pre-softmax logits → `match_vgg_logits.py`

In [ ]:
import glob
import numpy as np

def sniff(path):
    arr = np.load(path, allow_pickle=True)
    if arr.dtype == object:
        obj = arr.item() if arr.shape == () else arr
        if isinstance(obj, dict):
            for k in ('embeddings', 'features', 'vectors'):
                if k in obj:
                    return np.asarray(obj[k]).reshape(-1).shape[0]
    return np.asarray(arr).reshape(-1).shape[0]

for p in sorted(glob.glob('/content/*.npy')):
    print(f'{p}: dim={sniff(p)}')

## 5. Match the ArcFace unknowns (512-d)

Cosine-match against the full gallery (celebrities + classmates). Edit the filenames below to match what the TA hands out.

In [ ]:
!python scripts/identify_unknowns.py /content/ID1.npy --top-k 5
!python scripts/identify_unknowns.py /content/ID2.npy --top-k 5
!python scripts/identify_unknowns.py /content/ID3_UPDATED.npy --top-k 5

## 6. Match the VGG19 unknowns (1000-d pre-softmax logits)

**Why a separate script:** the TA's VGG vectors are the 1000-d `predictions`-layer output (pre-softmax logits), not the 4096-d `fc2` features the rest of the pipeline uses. We extract matching 1000-d logits (`fc2 @ W + b`) from every image in the dataset.

**Why the *robust* variant (vs the original `match_vgg_logits.py`):** the robust matcher trains on **every image — Gallery and Probe combined**, including the noisy Lighting / Occlusion / Side conditions. That gives the Logistic Regression head far more variation per identity than the thin clean-only gallery, which is what we needed to push the correct ID into the top-1 for ID4 and ID5. We also report a Nearest-Neighbor fallback against the full pool so the cosine geometry is visible side-by-side with the LR confidences.

**`--classmates-only`:** drops the 10 original celebrities so the dense celebrity rows don't outweigh the sparse classmate ones. The TA's challenge vectors are always classmates.

In [ ]:
%%writefile scripts/robust_match_vgg_logits.py
"""Robust matcher for 1000-d VGG19 pre-softmax logits against the FULL dataset."""
from pathlib import Path
import argparse
import os
import sys
import cv2
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import DATA_ROOT, RANDOM_SEED
from src.feature_extraction import init_arcface, walk_image_tasks
from src.metadata import parse_base_identity

def _load_unknown(path: Path) -> np.ndarray:
    arr = np.load(path, allow_pickle=True)
    if arr.dtype == object:
        obj = arr.item() if arr.shape == () else arr
        if isinstance(obj, dict):
            for k in ("features", "embeddings", "vectors"):
                if k in obj:
                    return np.asarray(obj[k], dtype=np.float32).reshape(1, -1)
            raise ValueError(f"no recognized key in {path}; got {list(obj)}")
    return np.asarray(arr, dtype=np.float32).reshape(1, -1)

def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("vectors", nargs="+", type=Path)
    ap.add_argument("--top-k", type=int, default=5)
    ap.add_argument("--classmates-only", action="store_true")
    ap.add_argument("--center", action="store_true", default=True)
    ap.add_argument("--gpu", action="store_true")
    args = ap.parse_args()

    # Tell TensorFlow not to claim the whole GPU so InsightFace/ArcFace can also load.
    import tensorflow as tf
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        try:
            for gpu in physical_devices:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(e)

    from tensorflow.keras.applications.vgg19 import VGG19, preprocess_input
    from tensorflow.keras.models import Model

    print("Loading VGG19...")
    base = VGG19(weights="imagenet", include_top=True)
    fc2_model = Model(inputs=base.input, outputs=base.get_layer("fc2").output)
    W, b = base.get_layer("predictions").get_weights()

    print("Loading ArcFace...")
    app = init_arcface(use_gpu=args.gpu)

    tasks = list(walk_image_tasks(DATA_ROOT))
    print(f"Extracting 1000-d features from ALL {len(tasks)} images (Gallery + Probe)...")

    feats, labels = [], []
    for i, (split, root, name) in enumerate(tasks):
        if i % 50 == 0 and i > 0:
            print(f"  Processed {i}/{len(tasks)} images...")

        img = cv2.imread(os.path.join(root, name))
        if img is None: continue
        faces = app.get(img)
        if not faces: continue
        x1, y1, x2, y2 = faces[0].bbox.astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: continue
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (224, 224))
        batch = preprocess_input(np.expand_dims(resized, axis=0))
        fc2 = fc2_model.predict(batch, verbose=0)
        feats.append((fc2 @ W + b)[0].astype(np.float32))
        labels.append(parse_base_identity(os.path.basename(root), name))

    print("Extraction complete!")
    X_raw = np.vstack(feats)
    mean_vec = X_raw.mean(axis=0, keepdims=True) if args.center else 0.0
    X = normalize(X_raw - mean_vec, norm="l2")
    labels = np.array(labels)

    if args.classmates_only:
        keep = np.array([n.isdigit() for n in labels])
        X, labels = X[keep], labels[keep]

    print(f"Training robust Logistic Regression on {len(X)} images ({len(set(labels))} identities)...")
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")
    clf.fit(X, labels)

    def prep_unknown(path):
        return normalize(_load_unknown(path) - mean_vec, norm="l2")

    for path in args.vectors:
        u = prep_unknown(path)
        probs = clf.predict_proba(u)[0]
        order = probs.argsort()[::-1][:args.top_k]

        print(f"\n=== {path} ===")
        print("Logistic Regression Confidence:")
        for rank, j in enumerate(order, 1):
            tag = " <- predicted" if rank == 1 else ""
            print(f"  {rank}. {clf.classes_[j]:<20} prob={probs[j]:.4f}{tag}")

        sims = (u @ X.T)[0]
        nn_order = sims.argsort()[::-1][:args.top_k]
        print("\nNearest Neighbor Fallback (Cosine Sim vs FULL dataset):")
        seen = set()
        rank = 1
        for idx in nn_order:
            label = labels[idx]
            if label not in seen:
                print(f"  {rank}. {label:<20} max_sim={sims[idx]:.4f}")
                seen.add(label)
                rank += 1
            if rank > args.top_k: break

if __name__ == "__main__":
    main()


In [ ]:
!python scripts/robust_match_vgg_logits.py /content/ID4.npy /content/ID5_UPDATED.npy --classmates-only --top-k 5
